In [1]:
%logstart -o notebook_log1.txt append

Activating auto-logging. Current session state plus future input saved.
Filename       : notebook_log1.txt
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : False
State          : active


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_DIR))

from load_store_data import load_data,store_documents
from models import list_available_chat_models
from build_documents import build_documents
from evaluation import evaluation_queries,evaluate_retrieval,hit_rate, mean_reciprocal_rank
from embeddings import create_embeddings
from rag import TechnicalGermanRAG
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from google import genai
import random
import os

import chromadb
load_dotenv()

/Users/eli/vorstellungsgesprach/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
client_gemini = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [4]:
pwd()

'/Users/eli/vorstellungsgesprach/notebook'

available_models = list_available_chat_models(client_gemini)
available_models

In [5]:
def run_pipeline(
    evaluation_queries,
    data_path="../data/raw/topics.json",
    chroma_path="../data/processed/chroma_db",
    collection_name="concepts_de",
    evaluation_output_path="../outputs/evaluation_queries.txt",
    embedding_model_name=(
        "sentence-transformers/"
        "paraphrase-multilingual-MiniLM-L12-v2"
    ),
):
    topics = load_data(data_path)

    documents = build_documents(topics)

    model, vectors = create_embeddings(
        documents=documents,
        model_name=embedding_model_name,
    )

    collection = store_documents(
        documents=documents,
        vectors=vectors,
        chroma_path=chroma_path,
        collection_name=collection_name,
    )

    retrieval_results = evaluate_retrieval(
        evaluation_queries=evaluation_queries,
        model=model,
        collection=collection,
        output_path=evaluation_output_path,
    )

    return {
        "topics": topics,
        "documents": documents,
        "model": model,
        "vectors": vectors,
        "collection": collection,
        "retrieval_results": retrieval_results,
    }

In [6]:
pipeline = run_pipeline(
    evaluation_queries=evaluation_queries
)

Batches: 100%|██████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.40it/s]


In [7]:
topics = pipeline["topics"]
documents = pipeline["documents"]
model = pipeline["model"]
vectors = pipeline["vectors"]
collection = pipeline["collection"]
retrieval_results = pipeline["retrieval_results"]

In [8]:
print(
    f"Documents indexed: "
    f"{pipeline['collection'].count()}"
)

print(
    pipeline["collection"].get()["ids"]
)

Documents indexed: 18
['q001', 'q002', 'q003', 'q004', 'q005', 'q006', 'q007', 'q008', 'q009', 'q010', 'q011', 'q012', 'q013', 'q014', 'q015', 'q016', 'q017', 'q018']


model = SentenceTransformer( "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

texts = [document["text"] for document in documents]

vectors = model.encode(
    texts,
    normalize_embeddings=True,
)

client_chroma = chromadb.PersistentClient(path="../data/processed/chroma_db")

collection = client_chroma.get_or_create_collection(
    name="concepts_de"
)

collection.add(
    ids=[doc["id"] for doc in documents],
    embeddings=vectors.tolist(),
    documents=[doc["text"] for doc in documents],
    metadatas=[doc["metadata"] for doc in documents],
)

query = "Welche Konzepte gehören zur binären Klassifikation?"

query_vector = model.encode(
    query,
    normalize_embeddings=True,
)

results = collection.query(
    query_embeddings=[query_vector.tolist()],
    n_results=3,
)

for doc, metadata, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(
        f"\n--- {metadata['topic']} "
        f"(distância: {distance:.3f}) ---"
    )
    print(f"Tags: {metadata.get('tags', '')}")
    print(doc)

In [9]:
retrieval_results = evaluate_retrieval(
    evaluation_queries=evaluation_queries,
    model=model,
    collection=collection,
    output_path="../outputs/evaluation_queries.txt",
)

In [10]:
hit_rate_at_1 = hit_rate(retrieval_results, k=1)
hit_rate_at_3 = hit_rate(retrieval_results, k=3)

print(f"Hit Rate@1: {hit_rate_at_1:.3f}")
print(f"Hit Rate@3: {hit_rate_at_3:.3f}")

Hit Rate@1: 0.704
Hit Rate@3: 0.870


In [11]:
for result in retrieval_results:
    first_id = result["retrieved_ids"][0]
    correct = first_id == result["expected_id"]

    print(f"\nConsulta: {result['query']}")
    print(f"Esperado: {result['expected_id']}")
    print(f"Primeiro resultado: {first_id}")
    print(f"Acertou: {correct}")


Consulta: Was bedeutet es, wenn eine Zielklasse viel seltener vorkommt als die andere?
Esperado: q001
Primeiro resultado: q001
Acertou: True

Consulta: Welches Problem entsteht bei einer sehr kleinen positiven Klasse?
Esperado: q001
Primeiro resultado: q001
Acertou: True

Consulta: Wie nennt man eine ungleichmäßige Verteilung der Zielklassen?
Esperado: q001
Primeiro resultado: q001
Acertou: True

Consulta: Wie lassen sich richtige und falsche Klassifikationen darstellen?
Esperado: q002
Primeiro resultado: q013
Acertou: False

Consulta: Was bedeuten True Positives und False Negatives?
Esperado: q002
Primeiro resultado: q002
Acertou: True

Consulta: Welche Tabelle enthält TP, TN, FP und FN?
Esperado: q002
Primeiro resultado: q018
Acertou: False

Consulta: Wie können ähnliche Kunden automatisch gruppiert werden?
Esperado: q003
Primeiro resultado: q003
Acertou: True

Consulta: Bei welchem Verfahren sollen Gruppen intern homogen sein?
Esperado: q003
Primeiro resultado: q003
Acertou: True



In [12]:
mrr = mean_reciprocal_rank(retrieval_results)

print(f"MRR: {mrr:.3f}")

MRR: 0.775


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

document_texts = [doc["text"] for doc in documents]
document_ids = [doc["id"] for doc in documents]

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
)

document_matrix = tfidf_vectorizer.fit_transform(document_texts)

def tfidf_search(query: str, n_results: int = 3) -> list[str]:
    query_vector = tfidf_vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector,
        document_matrix,
    )[0]

    ranked_positions = similarities.argsort()[::-1][:n_results]

    return [
        document_ids[position]
        for position in ranked_positions
    ]

tfidf_results = []

for item in evaluation_queries:
    retrieved_ids = tfidf_search(
        query=item["query"],
        n_results=min(3, len(documents)),
    )

    tfidf_results.append({
        "query": item["query"],
        "expected_id": item["expected_id"],
        "retrieved_ids": retrieved_ids,
    })

for result in tfidf_results:
    print(f"\nConsulta: {result['query']}")
    print(f"Esperado: {result['expected_id']}")
    print(f"Recuperados: {result['retrieved_ids']}")

In [13]:
vector_metrics = {
    "hit_rate_at_1": hit_rate(retrieval_results, k=1),
    "hit_rate_at_3": hit_rate(retrieval_results, k=3),
    "mrr": mean_reciprocal_rank(retrieval_results),
}

#tfidf_metrics = {
 #   "hit_rate_at_1": hit_rate(tfidf_results, k=1),
  #  "hit_rate_at_3": hit_rate(tfidf_results, k=3),
   # "mrr": mean_reciprocal_rank(tfidf_results),
#}

In [14]:
print("Busca vetorial")
print(f"Hit Rate@1: {vector_metrics['hit_rate_at_1']:.3f}")
print(f"Hit Rate@3: {vector_metrics['hit_rate_at_3']:.3f}")
print(f"MRR: {vector_metrics['mrr']:.3f}")

Busca vetorial
Hit Rate@1: 0.704
Hit Rate@3: 0.870
MRR: 0.775


In [15]:
ANSWER_PROMPT = """Du hilfst einer Person, sich auf ein Data-Science-Vorstellungsgespräch auf Deutsch vorzubereiten.

Basierend auf dem folgenden Konzept, beantworte die Nutzerfrage NICHT mit einem Fließtext. Gib stattdessen zurück:
1. Eine sehr kurze Einleitung (max. 1 Satz)
2. Die wichtigsten Ausdrücke (Phrasen) aus dem Konzept, die man sich zum Lernen merken sollte

Nutzerfrage: {query}

Konzept:
Thema: {topic}
Frage: {question_de}
Antwort: {answer_de}
Wichtige Ausdrücke: {phrases}

Antworte NUR mit einem JSON-Objekt, ohne Markdown-Formatierung:
{{"intro": "...", "phrases": ["...", "...", "..."]}}
"""

In [17]:
failed_at_1 = [
    result
    for result in retrieval_results
    if result["retrieved_ids"][0] != result["expected_id"]
]

print(f"Failed at 1: {len(failed_at_1)}")

for result in failed_at_1:
    print(f"\nQuery: {result['query']}")
    print(f"Expected: {result['expected_id']}")
    print(f"Retrieved: {result['retrieved_ids']}")

Failed at 1: 16

Query: Wie lassen sich richtige und falsche Klassifikationen darstellen?
Expected: q002
Retrieved: ['q013', 'q015', 'q002']

Query: Welche Tabelle enthält TP, TN, FP und FN?
Expected: q002
Retrieved: ['q018', 'q016', 'q002']

Query: Wie kann die Robustheit eines Modells auf kleinen Datensätzen getestet werden?
Expected: q004
Retrieved: ['q014', 'q017', 'q012']

Query: Wie vergleicht man Ereignisraten zweier Gruppen über die Zeit?
Expected: q005
Retrieved: ['q008', 'q007', 'q011']

Query: Was bedeutet ein Wert kleiner als 1 beim Vergleich von Behandlungs- und Kontrollgruppe?
Expected: q005
Retrieved: ['q006', 'q017', 'q005']

Query: Welches Maß berücksichtigt auch den Zeitpunkt eines Ereignisses, nicht nur ob es eintritt?
Expected: q005
Retrieved: ['q008', 'q005', 'q012']

Query: Was ist der Unterschied zwischen Odds Ratio und relativem Risiko?
Expected: q007
Retrieved: ['q006', 'q007', 'q005']

Query: Wie vergleicht man Überlebensverteilungen zwischen mehreren Gruppen?

In [16]:
rag = TechnicalGermanRAG(
    collection=collection,
    embed_model=model,
    llm_client=client_gemini,
    topics=topics,
    model=available_models,
    answer_prompt=ANSWER_PROMPT,
)

NameError: name 'available_models' is not defined

In [ ]:
query = "Was bedeutet es, wenn eine Zielklasse viel seltener vorkommt?"

In [ ]:
all_results = []

for model_name in available_models:
    try:
        result = rag.rag(
            query=query,
            model_name=model_name,
        )
        all_results.append(result)

    except Exception as error:
        print(f"Error with {model_name}: {error}")


with open("../outputs/resultados_comparacao.txt", "w", encoding="utf-8") as f:
    for i, result in enumerate(all_results, start=1):
        f.write(
            f"\n[{i}/{len(all_results)}] Modell: "
            f"{result['model']}\n"
        )
        f.write(f"Frage: {result['query']}\n")
        f.write(f"Thema erkannt: {result['topic']}\n")
        f.write(f"Intro: {result['intro']}\n")
        f.write("Phrasen:\n")

        for phrase in result["phrases"]:
            f.write(f"  • {phrase}\n")

        f.write("-" * 80 + "\n")

In [ ]:
selected_query = random.choice(evaluation_queries)

print(f"Selected query: {selected_query['query']}")
print(f"Expected ID: {selected_query['expected_id']}")

In [ ]:
query_randon = rag.compare_models(
    evaluation_queries=[selected_query],
    candidate_models=available_models,
)

In [ ]:
with open("../outputs/query_randon.txt", "w", encoding="utf-8") as f:
    for i, result in enumerate(query_randon, start=1):
        f.write(
            f"\n[{i}/{len(query_randon)}] Modell: "
            f"{result['model']}\n"
        )
        f.write(f"Frage: {result['query']}\n")
        f.write(f"Thema erkannt: {result['topic']}\n")
        f.write(f"Intro: {result['intro']}\n")
        f.write("Phrasen:\n")

        for phrase in result["phrases"]:
            f.write(f"  • {phrase}\n")

        f.write("-" * 80 + "\n")